# Aula 1 — Companion prático

## Agentes baseados em LLMs com LangChain e LangGraph

Este notebook acompanha a primeira aula sobre agentes baseados em LLMs.

Objetivo: transformar os conceitos dos slides em pequenos experimentos executáveis.

Tópicos:

1. O que é um LLM no notebook.
2. Como configurar `get_llm()`.
3. Como criar uma ferramenta.
4. Como um agente escolhe e chama ferramentas.
5. Como representar o loop agêntico em LangGraph.
6. Como observar a trajetória de execução.

O notebook foi escrito para alunos que ainda não conhecem LangChain nem LangGraph.

## 0. Instalação

Execute a célula abaixo apenas se estiver em um ambiente novo.

Ela instala:

- `langchain`
- `langgraph`
- `langchain-openai`
- `langchain-ollama`
- `python-dotenv`

O notebook usa um modelo real via OpenAI ou Ollama.

Antes de executar as células que chamam o LLM, configure o provider e as credenciais do ambiente.

In [16]:
# Execute apenas se precisar instalar as dependências.
%pip install -q langchain langgraph langchain-openai langchain-ollama python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração global

Escolha o provider que será usado pelo notebook:

```python
PROVIDER = "openai"   # ou "ollama"
```

Para OpenAI, defina a variável de ambiente `OPENAI_API_KEY`.

Para Ollama, mantenha um modelo local rodando, por exemplo:

```bash
ollama pull llama3.1
ollama serve
```

In [17]:
from __future__ import annotations

from typing import TypedDict, Annotated, Literal, Any
import operator
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROVIDER = "openai"  # "openai" ou "ollama"

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "llama3.1"

def load_env() -> Path | None:
    '''Carrega variáveis de ambiente a partir de um .env próximo ao notebook.'''
    candidates = [
        Path.cwd() / ".env",
        Path.cwd() / "src/genai/.env",
    ]

    for env_path in candidates:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path

    return None

ENV_PATH = load_env()
print(".env carregado de:", ENV_PATH or "nenhum arquivo encontrado")

def pretty(obj: Any) -> None:
    '''Imprime objetos Python de forma legível.'''
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

def pretty_messages(obj: Any) -> None:
    '''Imprime mensagens em formato amigável para leitura humana.'''
    messages = obj.get("messages") if isinstance(obj, dict) else obj

    if not isinstance(messages, list):
        print("Nenhuma lista de mensagens encontrada para exibição amigável.")
        return

    print("Mensagens (visão amigável):")
    print("=" * 60)

    for index, message in enumerate(messages, start=1):
        if isinstance(message, dict):
            role = message.get("role") or message.get("type") or "message"
            content = message.get("content", "")
            tool_calls = message.get("tool_calls")
        else:
            role = getattr(message, "type", message.__class__.__name__)
            content = getattr(message, "content", str(message))
            tool_calls = getattr(message, "tool_calls", None)

        print(f"[{index}] {str(role).upper()}")

        if isinstance(content, str):
            print(content)
        else:
            print(json.dumps(content, indent=2, ensure_ascii=False, default=str))

        if tool_calls:
            print("tool_calls:")
            print(json.dumps(tool_calls, indent=2, ensure_ascii=False, default=str))

        print("-" * 60)

.env carregado de: /Users/ebezerra/ailab/gcc1734/src/genai/.env


## 2. O que um LLM faz

Um LLM recebe um contexto e gera uma continuação provável.

Nos agentes, o LLM não é o sistema inteiro. Ele é o motor de raciocínio.

O restante do sistema precisa cuidar de:

- memória;
- ferramentas;
- validação;
- execução;
- observabilidade;
- condição de parada.

A função `get_llm()` abaixo centraliza a configuração do modelo.

In [18]:
def get_llm():
    '''Retorna um chat model configurado para o provider escolhido.'''
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError(
                "OPENAI_API_KEY nao encontrada. Verifique o arquivo .env em src/genai/.env "
                "ou exporte a variavel no ambiente do notebook."
            )
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0)

    if PROVIDER == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=OLLAMA_MODEL, temperature=0)

    raise ValueError(f"Provider desconhecido: {PROVIDER}")


llm = get_llm()
print("LLM:", llm)

LLM: profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x169f48450> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1696a2dd0> root_client=<openai.OpenAI object at 0x169f48dd0> root_async_client=<openai.AsyncOpenAI object at 0x169eeb890> model_name='gpt-4o-mini' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_

## 3. Primeiro contato com LangChain

LangChain fornece abstrações para trabalhar com modelos, mensagens, ferramentas e agentes.

Uma chamada simples a um LLM real teria a forma:

```python
llm.invoke("Explique o que é um agente baseado em LLM em uma frase.")
```

A célula abaixo faz essa chamada diretamente.

In [19]:
def call_llm(prompt: str) -> str:
    '''Chama o LLM configurado e retorna apenas o texto da resposta.'''
    response = llm.invoke(prompt)
    return response.content


resposta = call_llm("Explique o que é um agente baseado em LLM em uma frase.")
print(resposta)

Um agente baseado em LLM (Modelo de Linguagem de Grande Escala) é um sistema que utiliza modelos avançados de processamento de linguagem natural para entender, gerar e interagir com texto de forma autônoma, realizando tarefas específicas ou respondendo a consultas de maneira contextualizada.


## 4. Ferramentas

Uma ferramenta é uma função que o agente pode usar.

Do ponto de vista do LLM, a ferramenta precisa ter:

1. nome;
2. descrição;
3. esquema de parâmetros.

No LangChain, podemos expor uma função Python como ferramenta com `@tool`.

As ferramentas abaixo são independentes do provider do LLM.

In [20]:
from langchain_core.tools import tool

@tool
def check_stock(product_id: str) -> dict:
    '''Consulta o estoque atual de um produto.

    Use quando precisar saber quantas unidades de um produto existem em estoque.
    Retorna o identificador do produto e a quantidade disponível.
    '''
    fake_inventory = {
        "X": 15,
        "A123": 42,
        "B777": 0,
    }
    return {
        "product_id": product_id,
        "units": fake_inventory.get(product_id, 0),
    }


@tool
def check_supplier(product_id: str) -> dict:
    '''Consulta o prazo de reabastecimento de um produto.

    Use quando precisar saber em quantos dias um fornecedor consegue repor um produto.
    '''
    fake_suppliers = {
        "X": {"available": True, "lead_time_days": 5},
        "A123": {"available": True, "lead_time_days": 2},
        "B777": {"available": False, "lead_time_days": None},
    }
    return {
        "product_id": product_id,
        **fake_suppliers.get(product_id, {"available": False, "lead_time_days": None}),
    }


@tool
def create_order(product_id: str, quantity: int, priority: str = "normal") -> dict:
    '''Cria um pedido de compra para um produto.

    Use apenas quando já houver evidência de que o estoque é insuficiente.
    '''
    return {
        "order_id": f"PO-{product_id}-001",
        "product_id": product_id,
        "quantity": quantity,
        "priority": priority,
        "status": "created",
    }


tools = [check_stock, check_supplier, create_order]

for t in tools:
    print("Nome:", t.name)
    print("Descrição:", t.description.splitlines()[0])
    print("Args:", t.args)
    print("-" * 60)

Nome: check_stock
Descrição: Consulta o estoque atual de um produto.
Args: {'product_id': {'title': 'Product Id', 'type': 'string'}}
------------------------------------------------------------
Nome: check_supplier
Descrição: Consulta o prazo de reabastecimento de um produto.
Args: {'product_id': {'title': 'Product Id', 'type': 'string'}}
------------------------------------------------------------
Nome: create_order
Descrição: Cria um pedido de compra para um produto.
Args: {'product_id': {'title': 'Product Id', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}, 'priority': {'default': 'normal', 'title': 'Priority', 'type': 'string'}}
------------------------------------------------------------


## 5. Simulando uma chamada de ferramenta

O LLM não executa a ferramenta diretamente.

Ele propõe algo como:

```json
{"name": "check_stock", "args": {"product_id": "X"}}
```

O orquestrador valida e executa.

A célula abaixo implementa um mini-orquestrador.

In [21]:
tool_registry = {t.name: t for t in tools}

def execute_tool_call(tool_call: dict) -> Any:
    '''Executa uma chamada de ferramenta validada.'''
    name = tool_call["name"]
    args = tool_call.get("args", {})

    if name not in tool_registry:
        raise ValueError(f"Ferramenta desconhecida: {name}")

    return tool_registry[name].invoke(args)


tool_call = {
    "name": "check_stock",
    "args": {"product_id": "X"},
}

observation = execute_tool_call(tool_call)
pretty(observation)

{
  "product_id": "X",
  "units": 15
}


## 6. De ferramenta para agente

Um agente não segue uma sequência fixa.

Ele decide, durante a execução:

- chamar ferramenta;
- observar resultado;
- chamar outra ferramenta;
- responder;
- parar.

Com LangChain, podemos criar um agente com `create_agent` e inspecionar a troca de mensagens.

## 7. Agente real com LangChain

A célula abaixo usa a API atual de agentes do LangChain.

O objetivo didático é mostrar a estrutura:

1. definimos ferramentas;
2. criamos o agente;
3. fazemos uma pergunta;
4. o agente decide se usa ferramentas.

In [22]:
def run_langchain_agent():
    '''Executa um agente LangChain real.'''
    from langchain.agents import create_agent

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=(
            "Você é um agente de estoque. "
            "Use ferramentas para consultar dados antes de tomar decisões. "
            "Explique a decisão final de forma curta."
        ),
    )

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "O produto X tem demanda média de 8 unidades por dia. "
                    "O nível de segurança é 10 unidades. "
                    "Devo emitir pedido emergencial?"
                ),
            }
        ]
    })

    return result


agent_result = run_langchain_agent()
pretty(agent_result)
print()
pretty_messages(agent_result)

{
  "messages": [
    "content='O produto X tem demanda média de 8 unidades por dia. O nível de segurança é 10 unidades. Devo emitir pedido emergencial?' additional_kwargs={} response_metadata={} id='6c731010-ad12-4ca5-874f-b1024e98a737'",
    "content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 225, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a36426925d', 'id': 'chatcmpl-DouhjoZqOYU8GOlfJ4G15U7AXPdbP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ead77-633c-7010-9585-37509916a4a7-0' tool_calls=[{'name': 'check_stock', 'args': {'product_id': 'X'}, 'id': 'call_Q0zcxNgcCr6w7L92PJ1lxIKs', 'type': 'too

## 8. LangGraph: por que grafos?

LangChain ajuda a criar agentes.

LangGraph ajuda a controlar fluxos mais complexos.

Com LangGraph, representamos o agente como grafo:

- estado compartilhado;
- nós;
- arestas;
- roteamento condicional;
- ciclos.

A seguir, criaremos um grafo mínimo que implementa uma decisão de estoque.

In [23]:
from langgraph.graph import StateGraph, END

class InventoryState(TypedDict):
    product_id: str
    daily_demand: int
    safety_stock: int
    stock_units: int | None
    supplier_available: bool | None
    lead_time_days: int | None
    projected_stock: int | None
    decision: str | None
    trace: Annotated[list[str], operator.add]


def node_check_stock(state: InventoryState) -> dict:
    obs = check_stock.invoke({"product_id": state["product_id"]})
    return {
        "stock_units": obs["units"],
        "trace": [f"check_stock -> {obs}"],
    }


def node_check_supplier(state: InventoryState) -> dict:
    obs = check_supplier.invoke({"product_id": state["product_id"]})
    return {
        "supplier_available": obs["available"],
        "lead_time_days": obs["lead_time_days"],
        "trace": [f"check_supplier -> {obs}"],
    }


def route_after_supplier(state: InventoryState) -> Literal["project_stock", "final_answer"]:
    if state["supplier_available"]:
        return "project_stock"
    return "final_answer"


def node_project_stock(state: InventoryState) -> dict:
    projected = state["stock_units"] - state["daily_demand"] * state["lead_time_days"]
    return {
        "projected_stock": projected,
        "trace": [f"projected_stock -> {projected}"],
    }


def route_after_projection(state: InventoryState) -> Literal["create_order", "final_answer"]:
    if state["projected_stock"] < state["safety_stock"]:
        return "create_order"
    return "final_answer"


def node_create_order(state: InventoryState) -> dict:
    obs = create_order.invoke({
        "product_id": state["product_id"],
        "quantity": 100,
        "priority": "urgent",
    })
    return {
        "decision": "Pedido emergencial criado.",
        "trace": [f"create_order -> {obs}"],
    }


def node_final_answer(state: InventoryState) -> dict:
    if state["supplier_available"] is False:
        decision = "Fornecedor indisponível. Escalonar para decisão humana."
    elif state["projected_stock"] >= state["safety_stock"]:
        decision = "Pedido emergencial não é necessário."
    else:
        decision = "Pedido emergencial criado."

    return {
        "decision": decision,
        "trace": [f"final_answer -> {decision}"],
    }


graph_builder = StateGraph(InventoryState)

graph_builder.add_node("check_stock", node_check_stock)
graph_builder.add_node("check_supplier", node_check_supplier)
graph_builder.add_node("project_stock", node_project_stock)
graph_builder.add_node("create_order", node_create_order)
graph_builder.add_node("final_answer", node_final_answer)

graph_builder.set_entry_point("check_stock")
graph_builder.add_edge("check_stock", "check_supplier")
graph_builder.add_conditional_edges(
    "check_supplier",
    route_after_supplier,
    {
        "project_stock": "project_stock",
        "final_answer": "final_answer",
    },
)
graph_builder.add_conditional_edges(
    "project_stock",
    route_after_projection,
    {
        "create_order": "create_order",
        "final_answer": "final_answer",
    },
)
graph_builder.add_edge("create_order", END)
graph_builder.add_edge("final_answer", END)

inventory_graph = graph_builder.compile()

initial_state = {
    "product_id": "X",
    "daily_demand": 8,
    "safety_stock": 10,
    "stock_units": None,
    "supplier_available": None,
    "lead_time_days": None,
    "projected_stock": None,
    "decision": None,
    "trace": [],
}

final_state = inventory_graph.invoke(initial_state)
pretty(final_state)

{
  "product_id": "X",
  "daily_demand": 8,
  "safety_stock": 10,
  "stock_units": 15,
  "supplier_available": true,
  "lead_time_days": 5,
  "projected_stock": -25,
  "decision": "Pedido emergencial criado.",
  "trace": [
    "check_stock -> {'product_id': 'X', 'units': 15}",
    "check_supplier -> {'product_id': 'X', 'available': True, 'lead_time_days': 5}",
    "projected_stock -> -25",
    "create_order -> {'order_id': 'PO-X-001', 'product_id': 'X', 'quantity': 100, 'priority': 'urgent', 'status': 'created'}"
  ]
}


## 9. Lendo a trajetória

A execução acima não é apenas uma resposta final.

Ela tem uma trajetória:

1. consultar estoque;
2. consultar fornecedor;
3. projetar estoque;
4. decidir o caminho;
5. criar pedido ou encerrar.

Esse é o ponto central dos agentes: avaliamos o processo, não apenas a saída.

In [24]:
for step, event in enumerate(final_state["trace"], start=1):
    print(f"{step}. {event}")

print()
print("Decisão:", final_state["decision"])

1. check_stock -> {'product_id': 'X', 'units': 15}
2. check_supplier -> {'product_id': 'X', 'available': True, 'lead_time_days': 5}
3. projected_stock -> -25
4. create_order -> {'order_id': 'PO-X-001', 'product_id': 'X', 'quantity': 100, 'priority': 'urgent', 'status': 'created'}

Decisão: Pedido emergencial criado.


## 10. Exercício 1

Modifique o estado inicial para testar o produto `A123`.

Perguntas:

1. O grafo cria pedido emergencial?
2. Por quê?
3. Qual nó toma essa decisão?
4. O LLM foi necessário nesse exemplo?

Use a célula abaixo.

In [25]:
exercise_state = {
    "product_id": "A123",
    "daily_demand": 8,
    "safety_stock": 10,
    "stock_units": None,
    "supplier_available": None,
    "lead_time_days": None,
    "projected_stock": None,
    "decision": None,
    "trace": [],
}

exercise_result = inventory_graph.invoke(exercise_state)

for step, event in enumerate(exercise_result["trace"], start=1):
    print(f"{step}. {event}")

print()
print("Decisão:", exercise_result["decision"])

1. check_stock -> {'product_id': 'A123', 'units': 42}
2. check_supplier -> {'product_id': 'A123', 'available': True, 'lead_time_days': 2}
3. projected_stock -> 26
4. final_answer -> Pedido emergencial não é necessário.

Decisão: Pedido emergencial não é necessário.


## 11. Exercício 2

Crie uma nova ferramenta chamada `notify_manager`.

Ela deve receber:

- `product_id`;
- `message`.

Depois, modifique o grafo para chamar essa ferramenta quando o fornecedor estiver indisponível.

Sugestão de caso: produto `B777`.

Este exercício mostra que agentes reais precisam lidar com exceções operacionais.

In [26]:
@tool
def notify_manager(product_id: str, message: str) -> dict:
    '''Notifica o gerente sobre uma situação que exige decisão humana.'''
    return {
        "product_id": product_id,
        "message": message,
        "status": "manager_notified",
    }


notify_manager.invoke({
    "product_id": "B777",
    "message": "Fornecedor indisponível. Avaliar alternativa manualmente.",
})

{'product_id': 'B777',
 'message': 'Fornecedor indisponível. Avaliar alternativa manualmente.',
 'status': 'manager_notified'}

## 12. O que este notebook mostrou

1. Um LLM sozinho não é um agente completo.
2. Ferramentas conectam o agente ao mundo.
3. O LLM propõe chamadas. O orquestrador executa.
4. LangChain oferece abstrações de modelos, ferramentas e agentes.
5. LangGraph explicita o controle: estado, nós, arestas e roteamento.
6. A trajetória importa tanto quanto a resposta final.

Para a próxima aula, a pergunta natural é:

> Como dar memória ao agente?

Isso leva a embeddings, busca semântica e RAG.